In [1]:
import sys
print(sys.executable)

c:\Users\richi\Projects\Knowledge-Engineering\.venv\Scripts\python.exe


Task: Help the trailer choose a route and driving action between towns. 
Goal:  Reach the destination safely, with minimal distance or time, while obeying road rules and condtions

Relevant Knowledge: 
- directed graph and distances
- Road type
- Road Conditions
- Traffic rules


Vocabulary


In [13]:
#create list of all routes and the distanes between them
roads = [
    ("Nairobi", "Ruiru", 29), ("Ruiru", "Thika", 26), ("Nairobi", "Mai Mahiu", 53),
    ("Thika", "Murang'a", 40), ("Murang'a", "Sagana", 25), ("Sagana", "Karatina", 25),
    ("Karatina", "Nyeri", 26), ("Nyeri", "Nanyuki", 61), ("Sagana", "Embu", 37),
    ("Mai Mahiu", "Naivasha", 35), ("Mai Mahiu", "Narok", 89), ("Naivasha", "Nakuru", 70),
    ("Nakuru", "Molo", 47), ("Nakuru", "Nyahururu", 64), ("Molo", "Kericho", 65),
    ("Kericho", "Bomet", 73), ("Bomet", "Narok", 82), ("Kericho", "Sondu", 45),
    ("Kericho", "Ahero", 62), ("Sondu", "Kisumu", 52), ("Ahero", "Kisumu", 22),
    ("Kisumu", "Homa Bay", 105), ("Homa Bay", "Kisii", 56), ("Bomet", "Kisii", 83),
    ("Kisii", "Migori", 66),
]

#creates the graph 
graph = {}

# goes through every route and splits them into Town A and Town B
for a, b, km in roads:
    if a not in graph:
        graph[a] = {}
    # a -> b
    graph[a][b] = km

    if b not in graph:
        graph[b] = {}
    #b -> a
    graph[b][a] = km



rules = [ 
    ("Raining", "Wet"),
    ("Wet", "Slippery"),
    ("Slippery", "ReduceSpeed"),
    ("Roadworks","Closed"),
    ("Flooded", "Closed"),
]


facts = set()



In [14]:
# Tell and Ask 
def tell(predicate, road):
    facts.add((predicate, road))
    inference_engine()

def ask(predicate, road):
    return (predicate, road) in facts 


In [15]:
#inference engine uses forward chaining 

def inference_engine():
    changed = True
    while changed:
        changed = False
        for a in graph:
            for b in graph[a]:
                road = (a, b)
                for premise, conclusion in rules:
                    if (premise, road) in facts and (conclusion, road) not in facts:
                        facts.add((conclusion, road))
                        changed = True

In [16]:
def uniform_cost_search(start, goal):
    frontier = {start: (0, [start])}   # defines waiting towns with their cost so far and the path so far
    explored = set()                   # towns already expanded
    step = 0
 
    while frontier:
        # picks the frontier town with the LOWEST cost so far
        town = min(frontier, key=lambda t: frontier[t][0]) #gets the town with the cheapest cost
        cost, path = frontier.pop(town)
        explored.add(town) # adds the town to the list of explored towns
        
 
        # checks whether the town chosen is the goal (goal test)
        if town == goal:
            return cost, path
 
        # expand every neighbouring town
        for neighbour, km in graph[town].items():
            if neighbour in explored:
                continue                       # already finished with this town
            if ask("Closed", (town, neighbour)):
                        continue
            new_cost = cost + km
            # add it, or replace the old entry if this route is cheaper
            if neighbour not in frontier or new_cost < frontier[neighbour][0]:
                frontier[neighbour] = (new_cost, path + [neighbour])
 
        
        shown = sorted(frontier.items(), key=lambda item: item[1][0])
        
 
    return None, None   # no route exists

In [17]:
print(uniform_cost_search("Nairobi","Kisumu"))

(354, ['Nairobi', 'Mai Mahiu', 'Naivasha', 'Nakuru', 'Molo', 'Kericho', 'Ahero', 'Kisumu'])


In [23]:
def agent(start,goal, percepts):
    #tells the agent the percepts
    for predicate, road in percepts:
        tell(predicate,road)

    cost, path = uniform_cost_search(start,goal)
    if path is None:
        return "No safe route available"

    advice = []

    for i in range(len(path) - 1):
        road = (path[i], path[i+1])
        action = "reduce speed" if ask("ReduceSpeed", road) else "drive normally"
        advice.append(f"{road[0]} -> {road[1]}: {action}")

    # tell the agent what action was chosen 
    tell("chosen", tuple(path))

    return f"Route: {' -> '.join(path)} ({cost} km)\n  " + "\n  ".join(advice)
    
    

In [ ]:
# give the agent the percepts
percepts = [
    ("Raining", ("Kericho", "Ahero")),
    ("Roadworks", ("Molo","Kericho")),
]

print(agent("Nairobi", "Kisumu", percepts))

print("Facts in the KB now:")
for f  in facts:
    print(f)

Route: Nairobi -> Mai Mahiu -> Narok -> Bomet -> Kericho -> Ahero -> Kisumu (381 km)
  Nairobi -> Mai Mahiu: drive normally
  Mai Mahiu -> Narok: drive normally
  Narok -> Bomet: drive normally
  Bomet -> Kericho: drive normally
  Kericho -> Ahero: reduce speed
  Ahero -> Kisumu: drive normally
Facts in the KB now:
('Roadworks', ('Molo', 'Kericho'))
('Closed', ('Molo', 'Kericho'))
('chosen', ('Nairobi', 'Mai Mahiu', 'Narok', 'Bomet', 'Kericho', 'Ahero', 'Kisumu'))
('Raining', ('Kericho', 'Ahero'))
('Wet', ('Kericho', 'Ahero'))
('Slippery', ('Kericho', 'Ahero'))
('ReduceSpeed', ('Kericho', 'Ahero'))
